# utils\n
Generated from 00_common/utils.py for Databricks notebook import.\n

In [ ]:
# Common utility helpers for Databricks ingestion framework.

from __future__ import annotations

import hashlib
import json
import logging
import sys
import uuid
from datetime import datetime, timezone
from typing import Any, Dict

# --- Time Utilities ---
def utc_now_iso() -> str:
    """Get current UTC time in ISO format."""
    return datetime.now(timezone.utc).isoformat()

# --- ID Generation ---
def new_load_id() -> str:
    """Generate a new unique load ID."""
    return str(uuid.uuid4())

# --- Hashing Utilities ---
def row_hash(payload: Dict[str, Any]) -> str:
    """Generate a SHA256 hash for a dictionary payload."""
    canonical = json.dumps(payload, sort_keys=True, default=str)
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()

# --- Validation Utilities ---
def require_keys(payload: Dict[str, Any], required: list[str]) -> None:
    """Raise ValueError if any required keys are missing from payload."""
    missing = [k for k in required if k not in payload]
    if missing:
        raise ValueError(f"Missing required keys: {missing}")

# --- String Utilities ---
def truncate_str(value: str, max_chars: int = 500) -> str:
    """Truncate a string to max_chars, replacing newlines for single-line output."""
    msg = str(value).replace("\n", " ").strip()
    return msg[:max_chars]

# --- Logging Utilities ---
class _JsonFormatter(logging.Formatter):
    """Emit each log record as a single-line JSON object."""
    def format(self, record: logging.LogRecord) -> str:
        payload: Dict[str, Any] = {
            "ts": datetime.fromtimestamp(record.created, tz=timezone.utc).isoformat(),
            "level": record.levelname,
            "logger": record.name,
            "message": record.getMessage(),
        }
        if record.exc_info:
            payload["exc_info"] = self.formatException(record.exc_info)
        return json.dumps(payload, default=str)

def get_logger(name: str = "ingestion_framework") -> logging.Logger:
    """Return a logger that writes structured JSON lines to stdout.
    Safe to call multiple times with the same name — returns the existing
    logger without adding duplicate handlers.
    """
    logger = logging.getLogger(name)
    if logger.handlers:
        return logger
    logger.setLevel(logging.DEBUG)
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(_JsonFormatter())
    logger.addHandler(handler)
    logger.propagate = False
    return logger